# TEXAS — Google Colab Quickstart

**TEXAS** (`texas-psm`) is a Bayesian proxy system model for TEX₈₆/Scaled Ring Index paleothermometry.

This notebook walks through:
1. Installing the package and CmdStan
2. Downloading the pre-computed posteriors from Zenodo
3. **Forward prediction** — temperature → Scaled RI (pure Python, no Stan)
4. **Inverse reconstruction** — Scaled RI → temperature with full uncertainty (runs Stan)

---

> **What requires Stan?**
> | Task | Stan needed? |
> |------|-------------|
> | `predict_proxy_from_T()` — plot calibration curve | ❌ No |
> | `predict_T_from_proxyObs()` — reconstruct paleotemperature | ✅ Yes |
> | `get_posterior()` — re-fit the calibration from scratch | ✅ Yes |

---

In [ ]:
# ── Step 1a: Install TEXAS ─────────────────────────────────────────────────────
# PyPI (recommended — package is published):
!pip install -q texas-psm

# Or install the latest development version from GitHub:
# !pip install -q git+https://github.com/PaleoLipidRR/TEXAS.git

import TEXAS
print(f"TEXAS version: {TEXAS.__version__}")

### Step 1b — Install CmdStan

**Skip this step** if you only want to run forward prediction (`predict_proxy_from_T`).

Required for inverse reconstruction (`predict_T_from_proxyObs`). Takes ~5–10 minutes. Must be re-run each new Colab session.

In [ ]:
# ── Step 1b: Install CmdStan (only needed for inverse reconstruction) ──────────
# Skip this cell if you only want forward prediction (predict_proxy_from_T).
import cmdstanpy
cmdstanpy.install_cmdstan(version="2.36.0", progress=True)

# Re-initialise TEXAS so it picks up the newly installed CmdStan:
import importlib, TEXAS, TEXAS.utils.paths as _paths
_paths.CMDSTAN_DIR = _paths.find_cmdstan("2.36.0")
print(f"✅ CmdStan ready: {_paths.CMDSTAN_DIR}")

---

## Step 2 — Mount Google Drive (optional)

Mount your Drive if your proxy data or posteriors are stored there.
Skip this cell if you plan to upload files directly or use the Zenodo download below.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Set a base path for your data on Drive — change to your folder:
DRIVE_BASE = "/content/drive/MyDrive/texas"

import os
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f"Drive mounted. Working folder: {DRIVE_BASE}")

---

## Step 3 — Download the pre-computed posteriors

The forward calibration posteriors (`.nc` files) are required for both forward
and inverse predictions. Download them once from Zenodo — they are cached for
the session. If you mounted Google Drive above, point the cache there so you
don't re-download every session.

In [ ]:
import os
import TEXAS

# ── Option A: cache posteriors on Google Drive (survives session restarts) ─────
# Uncomment after mounting Drive in Step 2:
# os.environ["TEXAS_CACHE_DIR"] = f"{DRIVE_BASE}/posteriors"

# ── Option B: cache in ephemeral Colab storage (re-downloaded each session) ────
# Nothing to set — default cache is used automatically.

# ── Download all posteriors from Zenodo (~560 MB ZIP, extracted ~315 MB) ───────
TEXAS.download_all()

# ── Or download just the canonical posteriors (recommended) ────────────────────
# TEXAS.download_posteriors([
#     "gen_logi_fixed_hier_crtp_univ_priorApprox_SST_scaledRI_cren3",
#     "gen_logi_fixed_hier_crtp_multiv_priorApprox_eiv_SST_gdgt23ratio_no3_1.0_scaledRI_cren3",
# ])

---

## Step 4 — Forward prediction: temperature → Scaled RI

`predict_proxy_from_T()` evaluates the calibration curve at a range of
temperatures. **No Stan required** — this is pure Python using the loaded posterior.

Use this to visualise the calibration curve or to generate forward-model
predictions from a temperature ensemble.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from TEXAS import predict_proxy_from_T

temperatures = np.linspace(0, 35, 200)

# Temperature-only posterior (SST, canonical Scaled RI₀₋₃)
result = predict_proxy_from_T(
    temperatures=temperatures,
    posterior="gen_logi_fixed_hier_crtp_univ_priorApprox_SST_scaledRI_cren3",
    percentiles=[5, 25, 50, 75, 95],
)

# Plot calibration curve with uncertainty
fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(temperatures, result["p5"],  result["p95"], alpha=0.2,  label="5–95%")
ax.fill_between(temperatures, result["p25"], result["p75"], alpha=0.35, label="25–75%")
ax.plot(temperatures, result["p50"], lw=2, label="Median")
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Scaled RI (RI₀₋₃)")
ax.set_title("TEXAS forward calibration — temperature-only (SST)")
ax.legend()
plt.tight_layout()
plt.show()

---

## Step 5 — Inverse reconstruction: Scaled RI → temperature

`predict_T_from_proxyObs()` reconstructs paleotemperatures from observed
Scaled RI values. **Requires CmdStan** (Step 1b).

Replace the example data below with your own downcore or coretop Scaled RI
observations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from TEXAS import predict_T_from_proxyObs

# ── Load your data ─────────────────────────────────────────────────────────────
# Option A — from Google Drive (uncomment after Step 2):
# df = pd.read_csv(f"{DRIVE_BASE}/my_coretop_data.csv")

# Option B — upload interactively:
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0])

# Option C — example synthetic data (replace with your own):
np.random.seed(42)
n = 20
scaledRI_obs = np.random.uniform(0.35, 0.75, n)   # your downcore Scaled RI₀₋₃ values

# ── Run inverse reconstruction ─────────────────────────────────────────────────
result = predict_T_from_proxyObs(
    proxyObs       = scaledRI_obs,
    prior_mu_t     = 15.0,    # prior mean temperature (°C) — set to your best site estimate
    prior_sigma_t  = 10.0,    # prior uncertainty (°C) — use ~10 when uncertain
    fwd_posterior_name = "gen_logi_fixed_hier_crtp_univ_priorApprox_SST_scaledRI_cren3",
    temptype       = "SST",
)

# ── Plot results ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
idx = np.arange(n)
ax.fill_between(idx, result["p5"], result["p95"], alpha=0.25, label="5–95% CI")
ax.fill_between(idx, result["p25"], result["p75"], alpha=0.4, label="25–75% CI")
ax.plot(idx, result["p50"], "o-", ms=5, lw=1.5, label="Median")
ax.set_xlabel("Sample index")
ax.set_ylabel("Reconstructed SST (°C)")
ax.set_title("TEXAS inverse reconstruction — temperature-only model")
ax.legend()
plt.tight_layout()
plt.show()

# ── Summary table ──────────────────────────────────────────────────────────────
results_df = pd.DataFrame({
    "scaledRI":  scaledRI_obs,
    "T_p5":  result["p5"],
    "T_p25": result["p25"],
    "T_p50": result["p50"],
    "T_p75": result["p75"],
    "T_p95": result["p95"],
})
results_df.round(2).head(10)

---

## Step 6 — Save results

Save the reconstruction results back to Google Drive or download to your computer.

In [ ]:
# ── Option A: save to Google Drive ────────────────────────────────────────────
# output_path = f"{DRIVE_BASE}/texas_results.csv"
# results_df.to_csv(output_path, index=False)
# print(f"Saved to {output_path}")

# ── Option B: download to your computer ───────────────────────────────────────
results_df.to_csv("texas_results.csv", index=False)
from google.colab import files
files.download("texas_results.csv")

---

## Notes

| | Colab (free) | Colab Pro | Local / Docker |
|---|---|---|---|
| Setup time | ~10 min first session | ~10 min first session | One-time ~15 min build |
| CmdStan persistence | ❌ Re-install each session | ✅ Longer runtime | ✅ Pre-built in image |
| Posterior persistence | ✅ Cache on Google Drive | ✅ Cache on Google Drive | ✅ Local `data/` folder |
| CPUs for Stan | 2 | 2–8 | As many as your machine |
| Recommended for | Exploration, sharing | Repeated invT runs | Full analysis |

**Caching posteriors on Google Drive**: set `TEXAS_CACHE_DIR` to a Drive path before importing TEXAS (see Step 3). The ZIP (~560 MB) is downloaded once; subsequent sessions skip the download.

**Using the multivariate EIV model**: if you have GDGT-2/3 ratio and NO₃ data, use the EIV posterior for more accurate reconstructions:

```python
result = predict_T_from_proxyObs(
    proxyObs      = scaledRI_obs,
    prior_mu_t    = 15.0,
    prior_sigma_t = 10.0,
    fwd_posterior_name = "gen_logi_fixed_hier_crtp_multiv_priorApprox_eiv_SST_gdgt23ratio_no3_1.0_scaledRI_cren3",
    gdgt23ratio   = my_g23_array,
    no3           = my_no3_array,   # or: no3=10.0 to disable NO₃ correction
    temptype      = "SST",
)
```

**Citation**: Rattanasriampaipong et al. (in prep). *TEXAS: A proxy system model for TEX₈₆ paleothermometry.* AGU Paleoceanography and Paleoclimatology.